# 🧠 LangGraph Agent with Memory + Tools (Ollama)
# ⭐ Features
* Short-term memory using LangGraph Checkpointer
* Stores the agent’s active conversation state during execution.
* Long-term memory (SQLite)
* Saves important user facts & interactions for future sessions.
# Tool Calling
* Calculator tool
* Memory read/write tools
* Any custom tool integrated with LangChain / LangGraph
* Local LLM via Ollama
* Runs models like llama2, mistral, phi, etc., fully offline on your system.
* ReAct-style Agent Loop

#  Agent thinks → decides → calls tool → observes result → updates memory → responds.

In [4]:
from langgraph.graph import StateGraph,END,MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_ollama import ChatOllama
from langchain_core.tools import tool

import sqlite3
import json

In [5]:
# create llm 

In [6]:
llm = ChatOllama(
    model= 'llama3.1:8b',
    temperature=0
)

# long- term Memory setup (SQlite)

In [7]:
conn = sqlite3.connect("memory.db")
cursor = conn.cursor()

cursor.execute(""" 
 CREATE TABLE IF NOT EXISTS memory (
     user_id TEXT,
     key TEXT,
     value TEXT
 )
 """)
conn.commit()

# create memory and store the data 

In [8]:
def save_memory(user_id,key,value):
    cursor.execute(
        "INSERT INTO memory VALUES (?, ?,?)",
        (user_id,key,json.dumps(value))
    )
    conn.commit()

def load_memory(user_id):
    cursor.execute(
        "SELECT key, value FROM memory WHERE user_id=?",
        (user_id,)
    )
    rows = cursor.fetchall()

    memory = {}

    for k,v in rows:
        memory[k] = json.loads(v)

    return memory

# define Tools

# calculator Tool

In [9]:
@tool
def calculator(expression: str) -> str:
    """ Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Memory recall Tool

In [10]:
@tool
def recall(user_id: str, key: str) -> str:
    """ Retrieve user info."""
    memory = load_memory(user_id)
    return str(memory.get(key,"not found"))

# remember tool

In [11]:
@tool 
def remember(user_id: str, key: str,value: str) -> str:
    """ save user info """
    save_memory(user_id,key,value)
    return f"saved{key} = {value}"

# Bind tools to LLm 

In [12]:
tools = [calculator,remember,recall]
llm_with_tools = llm.bind_tools(tools)

# Agent node 

In [13]:
def agent_node(state: MessagesState, config):
    user_id = config["configurable"]["thread_id"]

    messages = state["messages"]

    # load long-term memory
    memory = load_memory(user_id)

    #System prompt with memory 
    system_msg = (
        "You are a smart assistant. \n"
        "Use tools when needed. \n"
        f"User info:{memory}"
    )

    final_messages = [("system",system_msg), *messages]

    response = llm_with_tools.invoke(final_messages)

# tool execution Node

In [14]:
def tool_node(state: MessagesState,config):
    last_messages = state["messages"][-1]

    tool_calls = last_messages.tool_calls

    results = []

    for call in tool_calls:
        tool_name = call["name"]
        args = call["args"]

        if tool_name == "calculator":
            result = calculator.invoke(args)
        elif tool_name == "remember":
            result = remember.invoke(args)
        elif tool_name == "recall":
            result = recall.invoke(args)


        results.append({
            "role":"tool",
            "content": result,
            "tool_call_id":call["id"]
        })
    return {"messages": results}

# conditional routing

In [15]:
def should_continue(state: MessagesState):
    last_message = state["messages"][-1]

    if hasattr(last_message,"too_calls") and last_message.tool_calls:
        return "tools"

    return "__end__"

# Build graph 

In [18]:
builder = StateGraph(MessagesState)

builder.add_node("agent",agent_node)
builder.add_node("tools",tool_node)

builder.set_entry_point("agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "__end__": "__end__"
    }
)
builder.add_edge("tools","agent")

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

# run Agent

In [25]:

config = {
    "configurable": {
        "thread_id": "Yash Coder"
    }
}

# ---- Store memory ----
graph.invoke(
    {"messages": [("user", "My name is yash coder 🧑‍💻")]},
    config
)

# ---- Recall memory ----
result = graph.invoke(
    {"messages": [("user", "What is my cute name?")]},
    config
)

# ---- Get final assistant message ----
result["messages"][-1].content

'What is my cute name?'